# Extended Lab: Survival Analysis in Python
### Kaplan-Meier, Exponential, and Cox Proportional Hazards Models

**Based on:** Huy H., *Building Statistical Models in Python* (2023), Ch. 13–14, and the accompanying
`1_kaplan_meier.ipynb`, `2_Exponential_Models.ipynb`, and `3_Cox_Proportional_Hazards.ipynb` notebooks.

This notebook extends the original three notebooks into a single, guided lab. It keeps every original
exercise, adds several new ones that the source material does not cover (parametric model comparison,
proportional-hazards diagnostics, multivariate Cox on a second dataset, restricted mean survival time),
and closes with discussion questions.

**Datasets used**
- `larynx` (lifelines) — 90 laryngeal cancer patients, univariate + stage/age covariates
- `heart` (R `survival` package, via `statsmodels`) — Stanford heart transplant waiting list, multivariate

**Learning objectives**
1. Estimate and interpret a non-parametric survival curve (Kaplan-Meier).
2. Fit and interpret a parametric survival model (Exponential), and compare it against alternatives.
3. Fit and interpret a semi-parametric regression model (Cox Proportional Hazards), including checking
   its core assumption.
4. Compare model families on the same data and articulate when each is (and is not) appropriate.

**Requirements**
```
pip install lifelines==0.27.4 statsmodels pandas matplotlib numpy --break-system-packages
```


## Part 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import datasets, plotting
from lifelines import KaplanMeierFitter, ExponentialFitter, WeibullFitter, LogNormalFitter, CoxPHFitter
from lifelines.statistics import logrank_test, proportional_hazard_test
from lifelines.utils import concordance_index

import statsmodels.api as sm

plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(42)


## Part 1 — Kaplan-Meier Estimator (Non-parametric)

### 1.1 Load and inspect the data
The `larynx` dataset records `time` (months survived after diagnosis), `age`, `death` (event indicator),
and three dummy columns `Stage_II`, `Stage_III`, `Stage_IV` (stage I is implied when all three are 0).

In [ ]:
data = datasets.load_larynx()
data.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(data.time, bins=15); axes[0].set_title('Distribution of survival time')
axes[1].hist(data.age, bins=20); axes[1].set_title('Distribution of age')
plt.tight_layout()

**Exercise 1.1** — What is the age range of the cohort? What fraction of patients experienced the event (death) rather than being censored? Answer in the cell below.

In [ ]:
age_range = data.age.max() - data.age.min()
event_rate = data.death.mean()
print(f'Age range: {age_range:.1f} years')
print(f'Event (death) rate: {event_rate:.1%}')

### 1.2 Fit the overall Kaplan-Meier curve

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(data.time, data.death, label='All patients')
kmf.survival_function_.head()

In [ ]:
fig, ax = plt.subplots(1)
kmf.survival_function_.plot(ax=ax)
ci = kmf.confidence_interval_survival_function_
ax.fill_between(ci.index, ci.values[:, 0], ci.values[:, 1], color='gray', alpha=0.3)
ax.set_xlabel('Months'); ax.set_ylabel('Survival probability')
ax.set_title('Kaplan-Meier survival curve — all patients');

**Exercise 1.2 (new)** — Report the median survival time and its confidence interval
using lifelines' built-in utilities, and compute the survival probability at 5 years (60 months).

In [ ]:
from lifelines.utils import median_survival_times
median = kmf.median_survival_time_
median_ci = median_survival_times(kmf.confidence_interval_)
print('Median survival time (months):', median)
print(median_ci)
print('S(60 months) estimate:', kmf.predict(60))

### 1.3 Compare groups: age (older vs. younger)

In [ ]:
split = 65
data['older'] = (data.age >= split).astype(int)
older = data[data.older == 1]
younger = data[data.older == 0]

older_kmf = KaplanMeierFitter().fit(older.time, older.death, label='older')
younger_kmf = KaplanMeierFitter().fit(younger.time, younger.death, label='younger')

fig, ax = plt.subplots(1)
older_kmf.plot(ax=ax, ci_show=True)
younger_kmf.plot(ax=ax, ci_show=True)
ax.set_title('Kaplan-Meier by age group');

**Exercise 1.3 (new)** — The book notes the confidence bands overlap and concludes age
"does not appear to be a strong factor." Confirm this quantitatively with a log-rank test.

In [ ]:
lr_age = logrank_test(older.time, younger.time, older.death, younger.death)
lr_age.print_summary()
print('\nConclusion: p =', round(lr_age.p_value, 3),
      '-> statistically significant at 5%?' , lr_age.p_value < 0.05)

### 1.4 Compare groups: cancer stage

In [ ]:
stages = data.columns[3:6] if 'older' in data.columns else data.columns[3:]
stage_cols = ['Stage_II', 'Stage_III', 'Stage_IV']
data['Stage_I'] = (1 - data[stage_cols].sum(axis=1)).clip(lower=0)

stage_data = {'stage_i': data[data.Stage_I == 1]}
for s in stage_cols:
    stage_data[s.lower()] = data[data[s] == 1]

fig, ax = plt.subplots(1, figsize=(9, 6))
fitters = {}
for name in sorted(stage_data):
    d = stage_data[name]
    f = KaplanMeierFitter().fit(d.time, d.death, label=name)
    fitters[name] = f
    f.plot(ax=ax, ci_show=False)
ax.set_title('Kaplan-Meier survival by cancer stage');

**Exercise 1.4 (new)** — Run a multi-group log-rank test (`multivariate_logrank_test`)
across all four stages simultaneously, rather than eyeballing the chart.

In [ ]:
from lifelines.statistics import multivariate_logrank_test

groups, durations, events = [], [], []
for name, d in stage_data.items():
    groups += [name] * len(d)
    durations += list(d.time)
    events += list(d.death)

result = multivariate_logrank_test(durations, groups, events)
result.print_summary()

> **Checkpoint:** Kaplan-Meier is descriptive — it tells you *whether* groups differ, not *why*, and it cannot quantify the effect of a continuous covariate like age directly.

## Part 2 — Exponential Model (Parametric)

### 2.1 Fit the exponential model to the whole cohort

In [ ]:
exf = ExponentialFitter().fit(data.time, data.death, label='Exponential')
exf.survival_function_.head()

In [ ]:
fig, ax = plt.subplots(1)
exf.plot_survival_function(ax=ax)
kmf.plot_survival_function(ax=ax, ci_show=False)
ax.set_title('Exponential vs. Kaplan-Meier survival curves');

**Exercise 2.1** — In your own words, why does the exponential curve look smooth while the Kaplan-Meier curve looks like stair steps? What does the fitted rate parameter $\lambda$ represent?

In [ ]:
print('Fitted lambda (event rate):', exf.lambda_)
print('Implied mean survival time (1/lambda):', 1 / exf.lambda_)

### 2.2 Group comparison: age

In [ ]:
exf_old = ExponentialFitter().fit(older.time, older.death, label='older')
exf_young = ExponentialFitter().fit(younger.time, younger.death, label='younger')

fig, ax = plt.subplots(1)
exf_old.plot_survival_function(ax=ax)
exf_young.plot_survival_function(ax=ax)
ax.set_title('Exponential survival — older vs younger');
print('Hazard ratio (older / younger):', exf_old.lambda_ / exf_young.lambda_)

### 2.3 Group comparison: stage

In [ ]:
fig, ax = plt.subplots(1, figsize=(9,6))
exp_fitters = {}
for name in sorted(stage_data):
    d = stage_data[name]
    f = ExponentialFitter().fit(d.time, d.death, label=name)
    exp_fitters[name] = f
    f.plot_survival_function(ax=ax)
ax.set_title('Exponential survival by stage');

### 2.4 (New) Is the constant-hazard assumption reasonable?

The exponential model assumes a **constant hazard over time** — a strong, often unrealistic assumption.
`lifelines` includes richer parametric families (Weibull, log-normal) that relax this assumption. We fit
all three to the same data and compare fit quality with AIC (lower is better).

In [ ]:
fitters_to_compare = {
    'Exponential': ExponentialFitter(),
    'Weibull': WeibullFitter(),
    'LogNormal': LogNormalFitter(),
}

results = {}
fig, ax = plt.subplots(1)
for name, f in fitters_to_compare.items():
    f.fit(data.time, data.death, label=name)
    results[name] = f.AIC_
    f.plot_survival_function(ax=ax)
kmf.plot_survival_function(ax=ax, ci_show=False, linestyle='--', color='black')
ax.set_title('Parametric model comparison vs. Kaplan-Meier (dashed)')

print('AIC by model (lower = better fit):')
for name, aic in sorted(results.items(), key=lambda kv: kv[1]):
    print(f'  {name:12s}: {aic:.2f}')

**Exercise 2.4** — Which parametric family fits best by AIC? Does the exponential model's constant-hazard assumption look justified for this dataset, based on how closely its curve tracks the Kaplan-Meier step function?

## Part 3 (New) — Cox Proportional Hazards on the Larynx Data

The original book chapter only demonstrates Cox PH on the Stanford heart transplant data. Here we apply
it to `larynx` so you practice building the design matrix yourself and interpreting a multivariate model
on a dataset you already know well from Parts 1–2.

In [ ]:
cox_data = data[['time', 'death', 'age', 'Stage_II', 'Stage_III', 'Stage_IV']].copy()

cph_larynx = CoxPHFitter()
cph_larynx.fit(cox_data, duration_col='time', event_col='death')
cph_larynx.print_summary()

**Exercise 3.1** — Which covariates are statistically significant at the 5% level? For the
significant stage variables, translate `exp(coef)` into a plain-English hazard-ratio statement (e.g.
"Stage IV patients have X times the hazard of Stage I patients, holding age constant").

In [ ]:
cph_larynx.plot()
plt.title('Larynx Cox PH — coefficients within 95% CI');

**Exercise 3.2 (new) — Check the proportional hazards assumption.**
Cox regression *assumes* hazard ratios between groups stay constant over the whole follow-up period.
This assumption is never tested in the original book chapter — we test it here with lifelines'
built-in Schoenfeld-residual check.

In [ ]:
cph_larynx.check_assumptions(cox_data, p_value_threshold=0.05, show_plots=True)

**Exercise 3.3** — Does any covariate fail the proportional-hazards check above? If so, what modeling options exist (e.g., stratification, time-varying covariates)? You do not need to implement a fix — just describe one option in a markdown cell.

## Part 4 — Cox Proportional Hazards on the Stanford Heart Transplant Data

### 4.1 Load data and set up train/test split

In [ ]:
heart = sm.datasets.get_rdataset('heart', package='survival').data
train_data = heart.iloc[0:171]
test_data = heart.iloc[171:]
heart.head()

### 4.2 Kaplan-Meier by transplant status (baseline comparison before modeling covariates)

In [ ]:
kmf_no = KaplanMeierFitter().fit(
    durations=heart.loc[heart.transplant == 0, 'stop'],
    event_observed=heart.loc[heart.transplant == 0, 'event'],
    label='no transplant')
kmf_yes = KaplanMeierFitter().fit(
    durations=heart.loc[heart.transplant == 1, 'stop'],
    event_observed=heart.loc[heart.transplant == 1, 'event'],
    label='transplant')

fig, ax = plt.subplots(1)
kmf_no.plot(ax=ax); kmf_yes.plot(ax=ax)
ax.set_xlabel('Days elapsed'); ax.set_ylabel('Survival probability')
ax.set_title('Kaplan-Meier Survival Estimates');

print('Non-transplant survival at day 300:', kmf_no.predict(300))
print('Transplant survival at day 300:', kmf_yes.predict(300))

### 4.3 Log-rank test

In [ ]:
lr_results = logrank_test(
    durations_A=heart.loc[heart.transplant == 0, 'stop'],
    durations_B=heart.loc[heart.transplant == 1, 'stop'],
    event_observed_A=heart.loc[heart.transplant == 0, 'event'],
    event_observed_B=heart.loc[heart.transplant == 1, 'event'])
lr_results.print_summary()

### 4.4 Fit the Cox Proportional Hazards model

In [ ]:
cph = CoxPHFitter()
cph.fit(df=train_data[['age', 'year', 'surgery', 'transplant', 'stop', 'event']],
        duration_col='stop', event_col='event')
cph.print_summary()

**Exercise 4.1** — Using `exp(coef)`, state the effect of `transplant` and `age` on hazard in plain English, matching the interpretation style from Chapter 14.

In [ ]:
cph.plot()
plt.title('Coefficients within 95% Confidence Intervals');

**Exercise 4.2 (new) — Check proportional hazards for the heart model.**

In [ ]:
cph.check_assumptions(train_data[['age', 'year', 'surgery', 'transplant', 'stop', 'event']],
                     p_value_threshold=0.05, show_plots=True)

### 4.5 Predict survival for all training patients and for the holdout patient

In [ ]:
cph.predict_survival_function(train_data[['age', 'year', 'surgery', 'transplant']]).plot(legend=False)
plt.xlabel('Survival Time'); plt.ylabel('Survival Probability'); plt.title('Survival Function for All Patients');

In [ ]:
cph.predict_survival_function(test_data[['age', 'year', 'surgery', 'transplant']]).plot()
plt.xlabel('Survival Time'); plt.ylabel('Survival Probability'); plt.title('Survival Function for Holdout');

**Exercise 4.3 (new) — Model discrimination.**
The book chapter reports a concordance value in `print_summary()` but never explains it. Compute and
interpret it here: a concordance of 0.5 is no better than random ranking, 1.0 is perfect ranking of who
dies first.

In [ ]:
c_index = concordance_index(train_data['stop'], -cph.predict_partial_hazard(train_data), train_data['event'])
print(f'Concordance index (train): {c_index:.3f}')

## Part 5 — Synthesis and Discussion

**Exercise 5.1** — Fill in the comparison table below based on everything you observed in Parts 1–4.

| Question | Kaplan-Meier | Exponential | Cox PH |
|---|---|---|---|
| Handles covariates? | | | |
| Assumes a hazard shape? | | | |
| Gives a single interpretable effect size? | | | |
| Assumption you must check before trusting it? | | | |
| Best used for... | | | |

**Exercise 5.2** — The `transplant` variable in the heart dataset is not randomized — patients who lived
long enough to receive a donor heart are, by construction, those who survived long enough to get one
("immortal time bias" / survivorship). Does the Cox PH hazard ratio for `transplant` in Part 4 represent
a **causal** effect of transplantation on survival? Why or why not?

**Exercise 5.3** — Both datasets used in this lab (larynx, n=90; heart, n=103) are small by modern
standards. What practical consequence does a small sample have for (a) Kaplan-Meier confidence bands,
(b) Cox PH coefficient standard errors, and (c) your confidence in the proportional-hazards check in
Exercises 3.2/4.2?

Write your answers in the cell below, or open `05_Detailed_Solutions.ipynb` to compare against a
worked answer key.


_Your answers here._